
# Week 6 — **Reinforcement Fine‑Tuning (RFT)**: Product Pricer

This notebook converts the Week 6 pricer pipeline from **supervised fine‑tuning (SFT)** to **reinforcement fine‑tuning (RFT)** using a **Python grader** that rewards numeric accuracy.

> **Why RFT?** We can directly reward **closeness to ground‑truth price** instead of only mimicking reference outputs.  
> **Colab**: Works in Google Colab. A GPU (T4) is **not required** for OpenAI fine‑tuning but is fine for local analytics.

**Created:** 2025-09-24 20:43 UTC


## 0) Setup
Install dependencies and create a `.env` file with your API keys in the same directory as this notebook.

In [1]:

#@title Install dependencies
# !pip -q install --upgrade openai wandb python-dotenv huggingface_hub matplotlib numpy pandas tqdm requests


## 1) Environment & API keys

Set your API keys in a `.env` file in the same directory as this notebook:

```
OPENAI_API_KEY=your_openai_key_here
ANTHROPIC_API_KEY=your_anthropic_key_here  
HF_TOKEN=your_huggingface_token_here
WANDB_API_KEY=your_wandb_key_here
```

- **OpenAI API key** required for RFT fine-tuning
- **Weights & Biases** optional for training charts
- **Anthropic & HuggingFace** optional for other models

In [2]:
# Load environment variables from .env file
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# Set up API keys from environment
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env') 
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')

# Optional W&B setup
USE_WANDB = bool(os.getenv("WANDB_API_KEY"))
if USE_WANDB:
    import wandb
    wandb.login()
    print("Weights & Biases logging enabled.")
else:
    print("Optional: set WANDB_API_KEY in your .env file to log training to Weights & Biases.")

# Sanity check
assert os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_API_KEY") != 'your-key-if-not-using-env', "OPENAI_API_KEY is required. Please set it in your .env file."
print("Environment ready.")

wandb: Currently logged in as: hafnium (hafnium49) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Weights & Biases logging enabled.
Environment ready.



## 2) Imports


In [3]:

import os, re, json, pickle, math, random, time, sys
from pathlib import Path
from typing import List, Dict, Any, Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import requests

from openai import OpenAI
client = OpenAI()

print("SDK ready.")


SDK ready.



## 3) Load `train.pkl` / `test.pkl`

If files are in Google Drive, mount and set `DATA_DIR` below.  
> **Note**: If you uploaded files earlier to ChatGPT, they may have expired—please re‑upload or use Drive.


In [4]:

#@title Locate your data
DATA_DIR = "."  #@param {type:"string"}
TRAIN_PKL = f"{DATA_DIR}/train.pkl"
TEST_PKL  = f"{DATA_DIR}/test.pkl"

if not Path(TRAIN_PKL).exists() or not Path(TEST_PKL).exists():
    print("train.pkl or test.pkl not found in DATA_DIR =", DATA_DIR)
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Drive mounted. Set DATA_DIR to /content/drive/MyDrive/<your-folder> and re-run this cell.")
    except Exception as e:
        print("Colab Drive not available or not needed:", e)

with open(TRAIN_PKL, "rb") as f:
    train = pickle.load(f)
with open(TEST_PKL, "rb") as f:
    test = pickle.load(f)

print(f"Loaded train: {len(train):,} | test: {len(test):,}")


Loaded train: 400,000 | test: 2,000



### 3.1 Project helpers (`Item`, `Tester`) or fallbacks
We try to use your existing `items.Item` and `testing.Tester`.  
If not found, we create minimal fallbacks so the notebook can run.


In [5]:

Item = None
Tester = None
try:
    from items import Item
    from testing import Tester
    print("Imported items.Item and testing.Tester.")
except Exception as e:
    print("Falling back to minimal helpers:", e)
    class _FallbackItem:
        def __init__(self, text, price):
            self.text = text
            self.price = float(price)
        def test_prompt(self):
            return f"How much does this cost?\n\n{self.text}\n\nPrice is $"
    if len(train) > 0 and not hasattr(train[0], "test_prompt"):
        def _wrap(dataset):
            wrapped = []
            for ex in dataset:
                if isinstance(ex, dict) and "text" in ex and "price" in ex:
                    wrapped.append(_FallbackItem(ex["text"], ex["price"]))
                else:
                    raise ValueError("Fallback mode expects dicts with keys: text, price")
            return wrapped
        train = _wrap(train)
        test  = _wrap(test)
        print("Wrapped dicts into fallback Item objects.")
    class _FallbackTester:
        @staticmethod
        def test(fn, dataset, max_n=None, name="model"):
            maes = []
            n = len(dataset) if max_n is None else min(max_n, len(dataset))
            for i in range(n):
                it = dataset[i]
                guess = fn(it)
                truth = float(it.price)
                err = abs(guess - truth)
                maes.append(err)
                color = "\x1b[92m" if err <= 50 else ("\x1b[93m" if err <= 150 else "\x1b[91m")
                print(f"{color}{i+1}: Guess: ${guess:.2f} Truth: ${truth:.2f} Error: ${err:.2f}\x1b[0m")
            print(f"\n{name} — MAE: ${np.mean(maes):.2f} | Median: ${np.median(maes):.2f}")
    Tester = _FallbackTester
print("Helpers ready.")


Imported items.Item and testing.Tester.
Helpers ready.



## 4) Utilities


In [6]:

def get_price(s: str) -> float:
    s = s.replace("$","").replace(",","") if s else ""
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0



## 5) RFT data: **messages + reference_answer**

For RFT, each JSONL line includes:
- `messages`: the prompt shown to the model
- `reference_answer`: ground truth string used by the grader


In [7]:

def rft_messages_for(item):
    # Keep it strict: numeric output only — reduces parsing errors
    system = "You estimate prices of items. Output only a number (USD), no symbols or text."
    user = item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": user}
    ]

def make_rft_jsonl(items, path):
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            obj = {
                "messages": rft_messages_for(it),
                "reference_answer": f"{float(it.price):.2f}"
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")
    print(f"Wrote {path} ({len(items):,} rows)")


In [8]:

#@title Split & write RFT JSONL
SEED = 42  #@param {type:"number"}
random.seed(SEED); np.random.seed(SEED)

# Small, fast iteration default; adjust if you like
N_TRAIN = 500   #@param {type:"number"}
N_VAL   = 50    #@param {type:"number"}

assert N_TRAIN <= len(train), "N_TRAIN exceeds available train size."
assert N_VAL   <= len(train), "N_VAL exceeds available train size."

idx = np.random.permutation(len(train))
fit_sel = [train[i] for i in idx[:N_TRAIN]]
val_sel = [train[i] for i in idx[N_TRAIN:N_TRAIN+N_VAL]]

OUT_DIR = Path("week6_rft")
OUT_DIR.mkdir(exist_ok=True, parents=True)
TRAIN_JSONL = OUT_DIR / "rft_train.jsonl"
VAL_JSONL   = OUT_DIR / "rft_val.jsonl"

make_rft_jsonl(fit_sel, TRAIN_JSONL)
make_rft_jsonl(val_sel, VAL_JSONL)

print("Data ready for RFT.")


Wrote week6_rft/rft_train.jsonl (500 rows)
Wrote week6_rft/rft_val.jsonl (50 rows)
Data ready for RFT.



## 6) Python grader (reward function)

We reward **closeness** via an exponential decay on absolute error.  
Returns **1.0** for exact match and decays toward **0** as error grows.


In [9]:

import inspect, math

def build_python_grader_payload(grader_fn):
    # The OpenAI API expects a function named `grade(sample, item)`.
    src = inspect.getsource(grader_fn)
    if not src.strip().startswith("def grade("):
        # rename if needed
        src = src.replace(grader_fn.__name__, "grade", 1)
    return {"type": "python", "source": src}

def price_grader(sample, item):
    """
    sample: { "output_text": "model raw output" }
    item:   { "reference_answer": "123.45" }
    return: float in [0,1]
    """
    def parse_num(s):
        if s is None: return None
        s = s.replace(",", "")
        m = re.search(r"[-+]?\d*\.?\d+", s)
        return float(m.group()) if m else None

    pred  = parse_num(sample.get("output_text", ""))
    truth = parse_num(item.get("reference_answer", ""))

    if pred is None or truth is None:
        return 0.0

    # Error scale: half‑life around 20% of truth or $25 minimum
    scale = max(25.0, 0.20 * max(1.0, truth))
    score = math.exp(-abs(pred - truth) / scale)

    # tiny penalty if non‑numeric fluff is present
    if re.search(r"[^\d\.\-+]", (sample.get("output_text","").strip())):
        score *= 0.98

    return float(max(0.0, min(1.0, score)))

GRADER_PAYLOAD = build_python_grader_payload(price_grader)
print("Grader payload ready.")


Grader payload ready.



### 6.1 (Optional) Validate grader with API
If this fails due to endpoint availability, it's safe to skip.


In [10]:

import json, os

try:
    headers = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}
    resp = requests.post(
        "https://api.openai.com/v1/fine_tuning/alpha/graders/validate",
        json={"grader": GRADER_PAYLOAD},
        headers=headers,
        timeout=30
    )
    if resp.ok:
        print("Grader validated ✓")
    else:
        print("Validation endpoint returned:", resp.status_code, resp.text[:200])
except Exception as e:
    print("Grader validation skipped (endpoint may be unavailable):", e)


Grader validated ✓



## 7) Upload files & launch **RFT**

> Pick an **RFT‑capable** base model. Start with a small number of epochs and iterate.


In [11]:

#@title Configure & launch RFT job
BASE_MODEL = "o4-mini-2025-04-16"  #@param {type:"string"}
EPOCHS     = 3                     #@param {type:"number"}
REASONING  = "medium"              #@param ["none","low","medium","high"]

# Upload files (purpose="fine-tune")
with open(TRAIN_JSONL, "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")
with open(VAL_JSONL, "rb") as f:
    val_file = client.files.create(file=f, purpose="fine-tune")

print("Uploaded:", train_file.id, val_file.id)

integrations = []
if USE_WANDB:
    integrations = [{"type":"wandb", "wandb": {"project": "gpt-pricer-rft"}}]

job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model=BASE_MODEL,
    seed=42,
    suffix=f"pricer-rft-{N_TRAIN}",
    method={
        "type": "reinforcement",
        "reinforcement": {
            "grader": GRADER_PAYLOAD,
            "hyperparameters": {
                "n_epochs": int(EPOCHS),
                "eval_interval": 5,
                "eval_samples": 3,
                "compute_multiplier": 1.0,
                "reasoning_effort": REASONING
            }
        }
    },
    integrations=integrations
)

print("RFT job created:", job.id, "| status:", job.status)


Uploaded: file-XPdvNvXLEyQJpTCpriNvHA file-3TuqpUo2ZL1ULabv66DJc2
RFT job created: ftjob-5mZalMokoAF7HN0LwjWbO7Vh | status: validating_files



## 8) Monitor job


In [16]:

def show_status(job_id, limit_events=10):
    j = client.fine_tuning.jobs.retrieve(job_id)
    print("Job:", j.id, "| status:", j.status, "| base:", j.model, "| fine_tuned:", j.fine_tuned_model)
    ev = client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=limit_events)
    for e in ev.data:
        print(f"- [{e.created_at}] {e.level}: {e.message}")
    return j

_ = show_status(job.id)


Job: ftjob-5mZalMokoAF7HN0LwjWbO7Vh | status: validating_files | base: o4-mini-2025-04-16 | fine_tuned: None
- [1758747723] info: Validating training file: file-XPdvNvXLEyQJpTCpriNvHA and validation file: file-3TuqpUo2ZL1ULabv66DJc2
- [1758747723] info: Created fine-tuning job: ftjob-5mZalMokoAF7HN0LwjWbO7Vh


In [ ]:
# Job: ftjob-5mZalMokoAF7HN0LwjWbO7Vh | status: validating_files | base: o4-mini-2025-04-16 | fine_tuned: None
# - [1758747723] info: Validating training file: file-XPdvNvXLEyQJpTCpriNvHA and validation file: file-3TuqpUo2ZL1ULabv66DJc2
# - [1758747723] info: Created fine-tuning job: ftjob-5mZalMokoAF7HN0LwjWbO7Vh


## 9) Use the fine‑tuned model (after **status = succeeded**)


In [13]:

def messages_for_inference(item):
    return rft_messages_for(item)

def gpt_rft_predict(item, model_name):
    rsp = client.chat.completions.create(
        model=model_name,
        messages=messages_for_inference(item),
        temperature=0,
        max_tokens=8,
        seed=42
    )
    txt = rsp.choices[0].message.content
    return get_price(txt)

# Try on a small slice first (cost-aware), then expand.
SMOKE = 8  # adjust to your budget
job_info = client.fine_tuning.jobs.retrieve(job.id)
ft_model = job_info.fine_tuned_model

if ft_model:
    print("Fine-tuned model:", ft_model)
    Tester.test(lambda it: gpt_rft_predict(it, ft_model), test, max_n=SMOKE, name="RFT (smoke)")
else:
    print("Model not ready. Re-run this cell after job succeeds.")


Model not ready. Re-run this cell after job succeeds.



## 10) Full evaluation (optional, cost-aware)


In [14]:

RUN_FULL = False  #@param {type:"boolean"}
if RUN_FULL:
    job_info = client.fine_tuning.jobs.retrieve(job.id)
    ft_model = job_info.fine_tuned_model
    if ft_model:
        Tester.test(lambda it: gpt_rft_predict(it, ft_model), test, name="RFT (full)")
    else:
        print("Model not ready yet.")
else:
    print("Set RUN_FULL=True to evaluate on full test set.")


Set RUN_FULL=True to evaluate on full test set.



## 11) Tips
- Keep output strict (numeric only) to make the reward meaningful.
- Start with small `N_TRAIN`, small `EPOCHS` and iterate quickly.
- Visualize with W&B (optional) to see reward curves, eval metrics.
- If the model learns to “game” the reward, refine the grader (e.g., penalize non‑numeric tokens more, or add item‑dependent scaling).
- Compare against your best **SFT** and **prompt‑engineering** baselines.
